##MAUNet

In [ ]:

import numpy as np
import glob
import cv2
import tensorflow as tf
import math
from skimage.metrics import structural_similarity as ssim
from matplotlib import pyplot as plt
from tensorflow.keras import layers


In [ ]:
#MAUNet
import tensorflow as tf
from tensorflow.keras.utils import plot_model
import keras

IMG_WIDTH = 128
IMG_HEIGHT = 128
IMG_CHANNELS = 1


#Build the model
inputs = tf.keras.layers.Input((IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS))

#Contraction path
c1 = tf.keras.layers.Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(inputs)
c1 = tf.keras.layers.Dropout(0.3)(c1)
c1 = tf.keras.layers.Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c1)
p1 = tf.keras.layers.MaxPooling2D((2, 2))(c1)
a1 = tf.keras.layers.AveragePooling2D((2,2))(c1)

c2 = tf.keras.layers.Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(p1)
c2 = tf.keras.layers.Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')((p1+c2)*0.5)
c2 = tf.keras.layers.Dropout(0.2)(c2)
c2 = tf.keras.layers.Conv2D(16, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')((c2+a1)*0.5)
p2 = tf.keras.layers.MaxPooling2D((2, 2))(c2)
a2 = tf.keras.layers.AveragePooling2D((2,2))(c2)

c3 = tf.keras.layers.Conv2D(16, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(p2)
c3 = tf.keras.layers.Conv2D(16, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')((p2+c3)*0.5)
c3 = tf.keras.layers.Dropout(0.1)(c3)
c3 = tf.keras.layers.Conv2D(16, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')((c3+a2)*0.5)

#Expansive path

u4 = tf.keras.layers.Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(c3)
u4 = tf.keras.layers.concatenate([((u4+p1)*0.5), c2])
c4 = tf.keras.layers.Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(u4)
c4 = tf.keras.layers.Dropout(0.2)(c4)
c4 = tf.keras.layers.Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c4)

u5 = tf.keras.layers.Conv2DTranspose(16, (2, 2), strides=(2, 2), padding='same')(c4)
u5 = tf.keras.layers.concatenate([((u5+inputs)*0.5), c1])
c5 = tf.keras.layers.Conv2D(16, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(u5)
c5 = tf.keras.layers.Dropout(0.1)(c5)
c5 = tf.keras.layers.Conv2D(16, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c5)

outputs = tf.keras.layers.Conv2D(1, (1, 1), activation='relu')(c5)

model = tf.keras.Model(inputs=[inputs], outputs=[outputs])
model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()
plot_model(model, show_shapes=True)


#Set EarlyStopping Criteria
earlystop=tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    min_delta=0,
    patience=20,
    verbose=0,
    mode='min',
    baseline=None,
    restore_best_weights=True
)
checkpoint=tf.keras.callbacks.ModelCheckpoint(
    filepath="checkmodel",
    monitor='val_loss',
    verbose=1,
    mode='min',
    save_best_only= True,
    )

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 128, 128, 1)]        0         []                            
                                                                                                  
 conv2d (Conv2D)             (None, 128, 128, 64)         640       ['input_1[0][0]']             
                                                                                                  
 dropout (Dropout)           (None, 128, 128, 64)         0         ['conv2d[0][0]']              
                                                                                                  
 conv2d_1 (Conv2D)           (None, 128, 128, 32)         18464     ['dropout[0][0]']             
                                                                                              

In [ ]:
import tensorflow as tf
from tensorflow.python.profiler.model_analyzer import profile
from tensorflow.python.profiler.option_builder import ProfileOptionBuilder

def get_flops(model):
  forward_pass = tf.function(model.call, input_signature=[tf.TensorSpec(shape=(1,) + model.input_shape[1:])])
  graph_info = profile(forward_pass.get_concrete_function().graph, options=ProfileOptionBuilder.float_operation())
  flops = graph_info.total_float_ops
  return flops


In [ ]:
flops = np.round(get_flops(model),4)
macs = flops / 2
print(f"MACs: {macs / 1e+9:,} G")
print(f"FLOPs: {flops / 1e+9:,} G")

MACs: 0.671842304 G
FLOPs: 1.343684608 G


##MAUNet Light

In [ ]:
#MAUNet Light

import tensorflow as tf
from tensorflow.keras.utils import plot_model
import keras

IMG_WIDTH = 128
IMG_HEIGHT = 128
IMG_CHANNELS = 1

#Build the model
inputs = tf.keras.layers.Input((IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS))

#Contraction path
c1 = tf.keras.layers.Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(inputs)
c1 = tf.keras.layers.Dropout(0.3)(c1)
c1 = tf.keras.layers.Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c1)
p1 = tf.keras.layers.MaxPooling2D((2, 2))(c1)
a1 = tf.keras.layers.AveragePooling2D((2,2))(c1)

c2 = tf.keras.layers.Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(p1)
c2 = tf.keras.layers.Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')((p1+c2)*0.5)
c2 = tf.keras.layers.Dropout(0.2)(c2)
c2 = tf.keras.layers.Conv2D(16, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')((c2+a1)*0.5)

u5 = tf.keras.layers.Conv2DTranspose(16, (2, 2), strides=(2, 2), padding='same')(c2)
u5 = tf.keras.layers.concatenate([(u5+inputs)*0.5, c1])
c5 = tf.keras.layers.Conv2D(16, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(u5)
c5 = tf.keras.layers.Dropout(0.1)(c5)
c5 = tf.keras.layers.Conv2D(16, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c5)
outputs = tf.keras.layers.Conv2D(1, (1, 1), activation='relu')(c5)

Student_Model = tf.keras.Model(inputs=[inputs], outputs=[outputs])
Student_Model.compile(optimizer='adam', loss='mean_squared_error')
Student_Model.summary()
plot_model(Student_Model, show_shapes=True)


#Set EarlyStopping Criteria
earlystop=tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    min_delta=0,
    patience=20,
    verbose=0,
    mode='min',
    baseline=None,
    restore_best_weights=True
  )
checkpoint=tf.keras.callbacks.ModelCheckpoint(
    filepath="checkmodel",
    monitor='val_loss',
    verbose=1,
    mode='min',
    save_best_only= True,
    )


Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_3 (InputLayer)        [(None, 128, 128, 1)]        0         []                            
                                                                                                  
 conv2d_16 (Conv2D)          (None, 128, 128, 64)         640       ['input_3[0][0]']             
                                                                                                  
 dropout_6 (Dropout)         (None, 128, 128, 64)         0         ['conv2d_16[0][0]']           
                                                                                                  
 conv2d_17 (Conv2D)          (None, 128, 128, 32)         18464     ['dropout_6[0][0]']           
                                                                                            

In [ ]:
flops = np.round(get_flops(Student_Model),4)
macs = flops / 2
print(f"MACs: {macs / 1e+9:,} G")
print(f"FLOPs: {flops / 1e+9:,} G")

MACs: 0.56365056 G
FLOPs: 1.12730112 G
